# Module 02: Pandas for Machine Learning
## Notebook 05: Combining Datasets and Time Series Manipulation

In enterprise machine learning, data is rarely stored in a single table. You must combine transaction databases with user demographic records and extract temporal patterns from time-stamped events.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Merge relational tables using `pd.merge()` with inner, left, right, and outer joins.
2. Concatenate data along rows or columns with `pd.concat()`.
3. Parse and manipulate timestamps using Pandas `to_datetime` and the `.dt` accessor.
4. Set datetime indices and resample data across daily, weekly, or monthly intervals.
5. Create rolling window moving averages and lag features for predictive forecasting.

In [1]:
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")

Pandas version: 3.0.6


### 1. Merging Relational Tables: `pd.merge()`

Types of Joins:
- `inner` (default): Retains only keys present in **both** tables.
- `left`: Retains all keys from the left table; missing matches in the right table become `NaN`.
- `right`: Retains all keys from the right table.
- `outer`: Retains all keys from both tables.

In [2]:
# Table 1: User Demographics
df_users = pd.DataFrame({
    'user_id': [1, 2, 3, 4],
    'name': ['Alex', 'Bailey', 'Casey', 'Drew'],
    'plan': ['Free', 'Premium', 'Premium', 'Free']
})

# Table 2: Activity Logs
df_activity = pd.DataFrame({
    'user_id': [1, 2, 2, 5],
    'login_count': [12, 45, 30, 8],
    'total_minutes': [120, 680, 420, 95]
})

# Inner Join
df_inner = pd.merge(df_users, df_activity, on='user_id', how='inner')
print("--- Inner Join (Matches only) ---\n", df_inner)

# Left Join (Preserve all registered users)
df_left = pd.merge(df_users, df_activity, on='user_id', how='left')
print("\n--- Left Join (All Users) ---\n", df_left)

--- Inner Join (Matches only) ---
    user_id    name     plan  login_count  total_minutes
0        1    Alex     Free           12            120
1        2  Bailey  Premium           45            680
2        2  Bailey  Premium           30            420

--- Left Join (All Users) ---
    user_id    name     plan  login_count  total_minutes
0        1    Alex     Free         12.0          120.0
1        2  Bailey  Premium         45.0          680.0
2        2  Bailey  Premium         30.0          420.0
3        3   Casey  Premium          NaN            NaN
4        4    Drew     Free          NaN            NaN


---
### 2. Concatenation: `pd.concat()`

- `axis=0` (default): Appends new rows (e.g., appending daily log files).
- `axis=1`: Appends new columns (e.g., combining preprocessed text features with tabular features).

In [3]:
# Simulating two monthly transaction batches
january_data = pd.DataFrame({'Sales': [100, 150], 'Units': [2, 3]}, index=['D1', 'D2'])
february_data = pd.DataFrame({'Sales': [120, 180], 'Units': [2, 4]}, index=['D3', 'D4'])

# Vertical Concatenation (axis=0)
all_sales = pd.concat([january_data, february_data], axis=0)
print("Vertically Concatenated Rows:\n", all_sales)

Vertically Concatenated Rows:
     Sales  Units
D1    100      2
D2    150      3
D3    120      2
D4    180      4


---
### 3. DateTime Processing & Feature Extraction

Time-based features (e.g., hour of day, day of week, is_weekend) are critical signals in fraud detection, recommendation systems, and demand forecasting.

In [4]:
dates = ['2026-03-01 09:15:00', '2026-03-05 14:30:00', '2026-03-07 22:45:00', '2026-03-08 08:00:00']
df_events = pd.DataFrame({'timestamp_str': dates, 'value': [10, 25, 40, 15]})

# Convert string to datetime
df_events['timestamp'] = pd.to_datetime(df_events['timestamp_str'])

# Extract temporal components using .dt accessor
df_events['year'] = df_events['timestamp'].dt.year
df_events['month'] = df_events['timestamp'].dt.month
df_events['day_name'] = df_events['timestamp'].dt.day_name()
df_events['hour'] = df_events['timestamp'].dt.hour
df_events['is_weekend'] = df_events['timestamp'].dt.dayofweek.isin([5, 6]).astype(int)

print("Engineered Temporal Features:\n", df_events[['timestamp', 'day_name', 'hour', 'is_weekend']])

Engineered Temporal Features:
             timestamp  day_name  hour  is_weekend
0 2026-03-01 09:15:00    Sunday     9           1
1 2026-03-05 14:30:00  Thursday    14           0
2 2026-03-07 22:45:00  Saturday    22           1
3 2026-03-08 08:00:00    Sunday     8           1


---
### 4. Time Series Resampling & Rolling Statistics

For time series models, setting a `DatetimeIndex` unlocks powerful window calculations:
- `.resample('D')`: Frequency aggregation (daily, weekly, monthly).
- `.rolling(window=N)`: Moving window statistics (e.g., 7-day moving average).
- `.shift(periods=1)`: Creates a **Lag Feature** ($y_{t-1}$) for forecasting $y_t$.

In [5]:
# Generating continuous daily time series data
time_index = pd.date_range(start='2026-01-01', periods=10, freq='D')
rng = np.random.default_rng(42)
df_ts = pd.DataFrame({'Daily_Revenue': rng.integers(100, 500, size=10)}, index=time_index)

# 1. 3-Day Rolling Moving Average
df_ts['Rolling_3D_Mean'] = df_ts['Daily_Revenue'].rolling(window=3).mean()

# 2. Lag Feature: Yesterday's revenue (critical for autoregressive ML models)
df_ts['Lag_1_Revenue'] = df_ts['Daily_Revenue'].shift(1)

print("Time Series with Rolling Average and Lag Feature:\n", df_ts)

Time Series with Rolling Average and Lag Feature:
             Daily_Revenue  Rolling_3D_Mean  Lag_1_Revenue
2026-01-01            135              NaN            NaN
2026-01-02            409              NaN          135.0
2026-01-03            361       301.666667          409.0
2026-01-04            275       348.333333          361.0
2026-01-05            273       303.000000          275.0
2026-01-06            443       330.333333          273.0
2026-01-07            134       283.333333          443.0
2026-01-08            378       318.333333          134.0
2026-01-09            180       230.666667          378.0
2026-01-10            137       231.666667          180.0


### Summary & Next Steps
In this notebook, you mastered:
- Relational joins (`inner`, `left`, `outer`) with `pd.merge()`.
- Vertical and horizontal dataset concatenation.
- Datetime parsing and component extraction.
- Constructing rolling averages and lag features for time-dependent ML.

**Next Notebook:** `06_feature_engineering_with_pandas.ipynb` — One-Hot Encoding, numerical binning, IQR outlier detection, and preparing clean $X, y$ datasets for Scikit-Learn.